# Notebook 01 — Data Exploration & Ingestion
**SuperStore Sales & Business Performance Analytics System**

This notebook covers:
- Loading the raw SuperStore CSV dataset
- Schema validation
- Initial EDA (Exploratory Data Analysis)
- Data quality assessment

In [ ]:
# ── Setup: Install dependencies if running in Databricks ──────────────────
# %pip install pandas numpy matplotlib seaborn pyyaml pyarrow

import sys
import os

# Adjust path if running locally
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='husl', font_scale=1.1)
print('Setup complete.')

## 1. Generate / Load Dataset

In [ ]:
# Generate synthetic dataset if not present
raw_path = os.path.join(PROJECT_ROOT, 'data', 'raw', 'superstore_sales.csv')

if not os.path.exists(raw_path):
    print('Generating synthetic dataset...')
    %run ../data/generate_dataset.py
else:
    print(f'Dataset found: {raw_path}')

df_raw = pd.read_csv(raw_path, low_memory=False)
print(f'Shape: {df_raw.shape}')
df_raw.head(5)

## 2. Schema Validation

In [ ]:
from src.ingestion.data_loader import DataLoader

loader = DataLoader()
df = loader.load(raw_path)
print('Schema validation passed!')
print(f'Rows: {len(df):,} | Columns: {len(df.columns)}')

## 3. Exploratory Data Analysis

In [ ]:
# Data types
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Shape:', df.shape, '===')

In [ ]:
# Null value heatmap
null_pct = (df.isnull().sum() / len(df) * 100).reset_index()
null_pct.columns = ['Column', 'Null %']
null_pct = null_pct[null_pct['Null %'] > 0]

if not null_pct.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(null_pct['Column'], null_pct['Null %'], color='coral')
    ax.set_xlabel('Null %')
    plt.title('Null Values by Column (%)')
    plt.tight_layout()
    plt.show()
else:
    print('No null values detected!')

In [ ]:
# Descriptive statistics on numeric columns
df[['Sales', 'Quantity', 'Discount', 'Profit']].describe().round(2)

In [ ]:
# Distribution of key numeric fields
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fields = ['Sales', 'Profit', 'Discount', 'Quantity']
for ax, field in zip(axes.flatten(), fields):
    ax.hist(df[field].dropna(), bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(f'Distribution: {field}')
    ax.set_xlabel(field)
    ax.set_ylabel('Count')
plt.suptitle('Numeric Field Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Categorical field value counts
cat_fields = ['Region', 'Category', 'Segment', 'Ship Mode']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, field in zip(axes.flatten(), cat_fields):
    vc = df[field].value_counts()
    ax.bar(vc.index, vc.values, color=sns.color_palette('husl', len(vc)))
    ax.set_title(f'Count by {field}')
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('Categorical Field Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of numeric fields
corr = df[['Sales', 'Quantity', 'Discount', 'Profit']].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
plt.title('Correlation Matrix — Numeric Fields')
plt.tight_layout()
plt.show()

In [ ]:
# Date range analysis
df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')
print(f'Order date range: {df["Order Date"].min().date()} → {df["Order Date"].max().date()}')
print(f'Total unique order dates: {df["Order Date"].nunique()}')
print(f'Total unique orders: {df["Order ID"].nunique():,}')
print(f'Total unique customers: {df["Customer ID"].nunique():,}')
print(f'Total unique products: {df["Product ID"].nunique():,}')